# Phase 1: data from the verified source, three test sets, and the complaint trends

Runs on a free Colab **CPU** runtime in about 20-30 minutes; no GPU is needed.
At the end it downloads `phase1_results.zip`: send that file back.

1. **Raw data.** `scripts/fetch_booking_515k.py` downloads the Booking 515K CSV
   from a pinned Hugging Face revision and refuses it unless its size and SHA-256
   match the Kaggle file the project was built on.
2. **Three splits of the same reviews** (`data/processed_v2*`): random, out-of-time
   (train on older reviews, test on newer) and unseen hotels.
3. **What each test measures.** `scripts/split_views.py` trains the classical
   candidates, picks the best on dev, and reports test macro-F1 with a bootstrap
   95% interval for each split.
4. **Which complaints are rising.** `scripts/complaint_trends.py` builds a SQLite
   database, runs the SQL in `analysis/complaint_trends/sql/`, compares the same
   months one year apart within the same hotels, and samples quotes for every
   topic it flags.

In [ ]:
!git clone --quiet --branch claude/funny-rubin-31xgtb https://github.com/krimits/hotel-review-nlp.git
%cd hotel-review-nlp
!git log --oneline -1
!pip install --quiet --no-deps -e .

## 1-2. Fetch the raw CSV (checked) and build the three splits

In [ ]:
!python scripts/fetch_booking_515k.py
!python -m reviewnlp.data.preprocess --config configs/baselines.yaml
!python -m reviewnlp.data.preprocess --config configs/baselines_time.yaml
!python -m reviewnlp.data.preprocess --config configs/baselines_hotel.yaml

## 3. The dev-selected classical model on each split

In [ ]:
!python scripts/split_views.py

## 4. Complaint trends: SQL, within-hotel changes, charts and quote samples

In [ ]:
!python scripts/complaint_trends.py --workers 2

In [ ]:
from IPython.display import Image, display

display(Image("runs/complaint_trends/within_hotel_change.png"))
display(Image("runs/complaint_trends/monthly_rates.png"))

## 5. Download the results (no review database, only results, manifests and quote samples)

In [ ]:
import glob
import zipfile

from google.colab import files

paths = [
    "docs/experiments/results/split_views.json",
    *sorted(glob.glob("data/processed_v2*/data_manifest.json")),
    *sorted(glob.glob("data/processed_v2*/label_stats.json")),
    *sorted(glob.glob("runs/classical_v2*/metrics.json")),
    "runs/complaint_trends/results.json",
    "runs/complaint_trends/quotes_for_review.json",
    "runs/complaint_trends/monthly_rates.png",
    "runs/complaint_trends/within_hotel_change.png",
    *sorted(glob.glob("runs/complaint_trends/csv/*.csv")),
]
with zipfile.ZipFile("phase1_results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in paths:
        archive.write(path)
print(f"{len(paths)} files")
files.download("phase1_results.zip")